# 04 - Main models (DataCo)
1. **Late-delivery classification** with LightGBM, on two populations:
   - *all orders* (as DataCo labels them)
   - *shipped only* (cancelled / suspected-fraud orders removed). This is the honest target: "will an order that ships arrive late?"
2. **ETA with prediction intervals** (shipped orders only): three interval methods compared by coverage and width.

Why two populations: cancelled orders (7,754, all paid by TRANSFER) are labelled "not late", so payment type partly
predicts *cancellation*, not delivery speed. The comparison shows how much of the model's edge is real.

Metrics: ROC-AUC, PR-AUC, **precision at 80% recall**, and **lift in the top 20%** of risk scores. F1 is not used for ranking models
because "flag everything" already scores 0.71 at a 55% late rate.

In [1]:
import sys; sys.path.append("..")
import yaml, joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier, LGBMRegressor

from src.data.load import load_dataco
from src.data.clean import clean_dataco
from src.features.build_features import (
    DATACO_CAT, DATACO_NUM, LEAKY_DATACO, DATACO_TARGET_CLF, DATACO_TARGET_REG,
    build_dataco_features, make_preprocessor, shipped_only, time_split)
from src.models.evaluate import (
    evaluate_binary, evaluate_regression, precision_at_recall, lift_at_top,
    threshold_for_recall, interval_metrics)
from src.models.train_eta import (
    empirical_group_intervals, apply_group_intervals, conformal_halfwidths, conformal_interval)
from src.utils.plotting import plot_roc_pr

cfg = yaml.safe_load(open("../config.yaml")); SEED = cfg["seed"]
tables = Path("../reports/tables"); tables.mkdir(parents=True, exist_ok=True)
figs = Path("../" + cfg["paths"]["figures"]); figs.mkdir(parents=True, exist_ok=True)
models_dir = Path("../" + cfg["paths"]["models"]); models_dir.mkdir(parents=True, exist_ok=True)

FEATS = DATACO_CAT + DATACO_NUM
assert not set(FEATS) & set(LEAKY_DATACO), "leaky column in features"
cat = [c for c in FEATS if c in DATACO_CAT]; num = [c for c in FEATS if c in DATACO_NUM]
Y = DATACO_TARGET_CLF

full = build_dataco_features(clean_dataco(load_dataco("../" + cfg["paths"]["dataco"])))
pops = {"all orders": full, "shipped only": shipped_only(full)}
pd.DataFrame({k: {"rows": len(v), "late_rate": round(v[Y].mean(), 3)} for k, v in pops.items()}).T

,rows,late_rate
all orders,180519.0,0.548
shipped only,172765.0,0.573


## 1. Late-delivery classification
LightGBM is tuned with a small random search (8 configurations) scored by validation ROC-AUC. Test data is touched once, at the end.

In [2]:
SPACE = {"num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1],
         "n_estimators": [200, 400], "min_child_samples": [20, 50, 100]}
FIXED = dict(subsample=0.8, subsample_freq=1, colsample_bytree=0.8, class_weight="balanced",
             random_state=SEED, verbose=-1)

def score_row(population, model, p_test, y_test):
    return {"population": population, "model": model,
            "roc_auc": round(roc_auc_score(y_test, p_test), 3),
            "pr_auc": round(average_precision_score(y_test, p_test), 3),
            "precision@80%recall": round(precision_at_recall(y_test, p_test, 0.8), 3),
            "lift@top20%": round(lift_at_top(y_test, p_test, 0.2), 2)}

rows, probs, finals, best_params = [], {}, {}, {}
for pop, data in pops.items():
    train, val, test = time_split(data, cfg["split"]["train"], cfg["split"]["val"])
    ytr, yva, yte = train[Y], val[Y], test[Y]

    # mode-only baseline
    rate = train.groupby("Shipping Mode")[Y].mean()
    p_mode = test["Shipping Mode"].map(rate).fillna(ytr.mean()).to_numpy()
    rows.append(score_row(pop, "mode-only late rate", p_mode, yte)); probs[(pop, "mode-only")] = p_mode

    # random forest baseline
    rf = Pipeline([("prep", make_preprocessor(cat, num)),
                   ("clf", RandomForestClassifier(n_estimators=150, min_samples_leaf=20, class_weight="balanced",
                                                  n_jobs=-1, random_state=SEED))]).fit(train[FEATS], ytr)
    p_rf = rf.predict_proba(test[FEATS])[:, 1]
    rows.append(score_row(pop, "random forest", p_rf, yte)); probs[(pop, "random forest")] = p_rf

    # LightGBM: random search on validation ROC-AUC
    prep = make_preprocessor(cat, num).fit(train[FEATS])
    Xtr, Xva = prep.transform(train[FEATS]), prep.transform(val[FEATS])
    best, best_auc = None, -1
    for params in ParameterSampler(SPACE, n_iter=8, random_state=SEED):
        m = LGBMClassifier(**params, **FIXED).fit(Xtr, ytr)
        auc = roc_auc_score(yva, m.predict_proba(Xva)[:, 1])
        if auc > best_auc: best, best_auc = params, auc
    lgbm = Pipeline([("prep", make_preprocessor(cat, num)),
                     ("clf", LGBMClassifier(**best, **FIXED))]).fit(train[FEATS], ytr)
    p_lgb = lgbm.predict_proba(test[FEATS])[:, 1]
    rows.append(score_row(pop, "LightGBM (tuned)", p_lgb, yte)); probs[(pop, "LightGBM")] = p_lgb
    finals[pop], best_params[pop] = lgbm, {**best, "val_auc": round(best_auc, 4)}

clf_results = pd.DataFrame(rows).set_index(["population", "model"])
clf_results.to_csv(tables / "models_dataco_classification.csv")
print("best LightGBM params:", best_params)
clf_results

best LightGBM params: {'all orders': {'num_leaves': 15, 'n_estimators': 200, 'min_child_samples': 100, 'learning_rate': 0.03, 'val_auc': 0.7601}, 'shipped only': {'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 100, 'learning_rate': 0.1, 'val_auc': 0.7711}}


roc_auc  pr_auc  precision@80%recall  \
population   model                                                       
all orders   mode-only late rate    0.731   0.750                0.623   
             random forest          0.774   0.837                0.660   
             LightGBM (tuned)       0.775   0.839                0.658   
shipped only mode-only late rate    0.746   0.788                0.653   
             random forest          0.778   0.859                0.677   
             LightGBM (tuned)       0.779   0.860                0.677   

                                  lift@top20%  
population   model                             
all orders   mode-only late rate         1.64  
             random forest               1.69  
             LightGBM (tuned)            1.70  
shipped only mode-only late rate         1.65  
             random forest               1.69  
             LightGBM (tuned)            1.71

In [3]:
pop = "shipped only"; _, _, test_s = time_split(pops[pop], cfg["split"]["train"], cfg["split"]["val"])
plot_roc_pr(test_s[Y], {k[1]: v for k, v in probs.items() if k[0] == pop}, figs / "models_dataco_curves.png")
plt.figure(figsize=(11, 4)); plt.imshow(plt.imread(figs / "models_dataco_curves.png")); plt.axis("off"); plt.show()

C:\Users\avdhi\AppData\Local\Temp\ipykernel_24264\402394438.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.figure(figsize=(11, 4)); plt.imshow(plt.imread(figs / "models_dataco_curves.png")); plt.axis("off"); plt.show()


**How to read this**
- Compare *LightGBM* with *mode-only* inside each population. The gap is what everything except shipping mode adds.
- Compare the gap in *all orders* with the gap in *shipped only*. If it shrinks a lot when cancelled orders are removed, part of the
  model's edge was predicting cancellation through payment type, not lateness.
- *precision@80%recall* answers: "if we want to catch 80% of the late orders, how many of the flagged orders are really late?"
  A random flag gives about 0.55 (the base rate).

### Operating point for the early-warning rules
Threshold chosen on the **validation** split so that 80% of late orders are caught, then applied once to the test split.

In [4]:
pop = "shipped only"
train, val, test = time_split(pops[pop], cfg["split"]["train"], cfg["split"]["val"])
model = finals[pop]
thr = threshold_for_recall(val[Y], model.predict_proba(val[FEATS])[:, 1], 0.8)
r = evaluate_binary(test[Y], model.predict_proba(test[FEATS])[:, 1], thr)
print(f"threshold = {r['threshold']}  recall = {r['recall']}  precision = {r['precision']}  "
      f"share of orders flagged = {r['flagged_share']}  confusion matrix [[TN, FP], [FN, TP]] = {r['confusion_matrix']}")
joblib.dump({"model": model, "threshold": thr, "features": FEATS}, models_dir / "late_delivery_lgbm.joblib")

# what drives it (SHAP comes in notebook 06): top LightGBM gains
names = model.named_steps["prep"].get_feature_names_out()
imp = pd.Series(model.named_steps["clf"].booster_.feature_importance("gain"), index=names).sort_values(ascending=False)
(imp / imp.sum()).head(10).round(3)

threshold = 0.323  recall = 0.8481  precision = 0.6515  share of orders flagged = 0.75  confusion matrix [[TN, FP], [FN, TP]] = [[4211, 6774], [2268, 12662]]


num__Days for shipment (scheduled)    0.370
cat__Shipping Mode_Standard Class     0.156
num__order_hour                       0.148
cat__Shipping Mode_First Class        0.144
cat__Shipping Mode_Second Class       0.030
cat__Shipping Mode_Same Day           0.029
num__order_weekday                    0.017
num__order_month                      0.016
num__discount_ratio                   0.012
num__Sales                            0.009
dtype: float64

## 2. ETA with prediction intervals (shipped orders only)
Target `Days for shipping (real)`. Nominal coverage is 80% (`alpha = 0.2`). Three interval methods are compared on the test split.

In [5]:
data = pops["shipped only"]; yr = DATACO_TARGET_REG; ALPHA = 0.2
train, val, test = time_split(data, cfg["split"]["train"], cfg["split"]["val"])
G = "Shipping Mode"
kw = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=50, subsample=0.8,
          subsample_freq=1, colsample_bytree=0.8, random_state=SEED, verbose=-1)
def pipe(**extra): return Pipeline([("prep", make_preprocessor(cat, num)), ("reg", LGBMRegressor(**kw, **extra))])

point = pipe().fit(train[FEATS], train[yr])
mode_mean = train.groupby(G)[yr].mean()
pm = pd.DataFrame([
    {"model": "mean real days per mode", **evaluate_regression(test[yr], test[G].map(mode_mean))},
    {"model": "LightGBM", **evaluate_regression(test[yr], point.predict(test[FEATS]))},
]).set_index("model")
pm

,mae,rmse,r2
model,,,
mean real days per mode,0.9822,1.2672,0.3848
LightGBM,0.9721,1.2643,0.3876


In [6]:
res = {}
# 1) empirical per-mode quantiles
emp = empirical_group_intervals(train, yr, G, ALPHA)
dflt = (train[yr].quantile(ALPHA / 2), train[yr].quantile(1 - ALPHA / 2))
lo, hi = apply_group_intervals(emp, test[G], dflt)
res["empirical per-mode quantiles"] = interval_metrics(test[yr], lo, hi, test[G])

# 2) LightGBM quantile regression
qlo = pipe(objective="quantile", alpha=ALPHA / 2).fit(train[FEATS], train[yr])
qhi = pipe(objective="quantile", alpha=1 - ALPHA / 2).fit(train[FEATS], train[yr])
lo, hi = qlo.predict(test[FEATS]), qhi.predict(test[FEATS])
res["LightGBM quantile regression"] = interval_metrics(test[yr], np.minimum(lo, hi), np.maximum(lo, hi), test[G])

# 3) split conformal, per mode (calibrated on the validation split)
hw = conformal_halfwidths(val[yr], point.predict(val[FEATS]), val[G], ALPHA)
lo, hi = conformal_interval(point.predict(test[FEATS]), test[G], hw)
res["split conformal (per mode)"] = interval_metrics(test[yr], lo, hi, test[G])

interval_results = pd.DataFrame(res).T
interval_results.to_csv(tables / "models_dataco_eta_intervals.csv")
print("conformal half-widths (days):", {k: round(v, 2) for k, v in hw.items()})
interval_results[[c for c in interval_results.columns if c in ("coverage", "mean_width") or c.startswith("cov_")]]

conformal half-widths (days): {'First Class': 0.04, 'Same Day': 0.06, 'Second Class': 2.0, 'Standard Class': 2.0, '__all__': 1.98}


,coverage,mean_width,cov_First Class,cov_Same Day,cov_Second Class,cov_Standard Class
empirical per-mode quantiles,1.000,3.230,1.000,1.000,1.000,1.000
LightGBM quantile regression,0.929,3.245,0.978,0.938,0.704,0.991
split conformal (per mode),0.797,3.191,0.784,0.829,0.801,0.796


In [7]:
w = interval_results[[c for c in interval_results.columns if c.startswith("width_")]]
w.columns = [c.replace("width_", "") for c in w.columns]
w.T.plot.bar(figsize=(9, 4), title="Mean interval width (days) by shipping mode, nominal 80% coverage"); plt.ylabel("days"); plt.xticks(rotation=0); plt.show()
joblib.dump({"point": point, "q_lo": qlo, "q_hi": qhi, "conformal_halfwidths": hw, "features": FEATS}, models_dir / "eta_lgbm.joblib")

C:\Users\avdhi\AppData\Local\Temp\ipykernel_24264\2616978117.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  w.T.plot.bar(figsize=(9, 4), title="Mean interval width (days) by shipping mode, nominal 80% coverage"); plt.ylabel("days"); plt.xticks(rotation=0); plt.show()


['..\\models\\eta_lgbm.joblib']

In [8]:
sd = pops["shipped only"]
sd = sd[sd["Shipping Mode"] == "Same Day"]
print(sd.groupby("order_hour")["Days for shipping (real)"].agg(["mean", "size"]).round(2))

# does the rest of the model need the time features at all?
train, val, test = time_split(pops["shipped only"], cfg["split"]["train"], cfg["split"]["val"])
no_time = [c for c in FEATS if c not in ("order_hour", "order_weekday", "order_month")]
c2 = [c for c in no_time if c in DATACO_CAT]; n2 = [c for c in no_time if c in DATACO_NUM]
bp = {k: v for k, v in best_params["shipped only"].items() if k != "val_auc"}
m = Pipeline([("prep", make_preprocessor(c2, n2)), ("clf", LGBMClassifier(**bp, **FIXED))]).fit(train[no_time], train[Y])
print("test ROC-AUC without time features:", round(roc_auc_score(test[Y], m.predict_proba(test[no_time])[:, 1]), 3), "(with: 0.779)")

            mean  size
order_hour            
0            0.0   430
1            0.0   432
2            0.0   338
3            0.0   468
4            0.0   447
5            0.0   327
6            0.0   426
7            0.0   401
8            0.0   385
9            0.0   407
10           0.0   383
11           0.0   395
12           1.0   409
13           1.0   355
14           1.0   348
15           1.0   370
16           1.0   346
17           1.0   410
18           1.0   317
19           1.0   421
20           1.0   326
21           1.0   351
22           1.0   418
23           1.0   383


test ROC-AUC without time features: 0.743 (with: 0.779)


**How to read this**
- *Coverage* should be near 0.80, and *mean width* should be as small as possible while keeping it.
- Real days are whole numbers and uniform inside Second / Standard Class (2 to 6), so intervals there are about 2 to 4 days wide
  whatever the method. First Class is exactly 2 days, so its interval should have width 0.
- If three methods give about the same width, the interval is a property of the data (delivery times are random inside a mode), not of the model.

## Results to copy into `reports/experiment_log.md`
Late classification (shipped only): LightGBM ROC-AUC ..., PR-AUC ..., precision@80%recall ..., lift@top20% ...  vs mode-only ...
Operating point: threshold ..., recall ..., precision ..., flagged share ...
ETA: LightGBM MAE ... vs mode mean ...;  80% interval coverage ... width ...

## Takeaways (fill in)
1. 
2. 
3. 